# EcoTracker: Анализ и Классификация Экологического Влияния

## Полный pipeline анализа данных и обучения моделей машинного обучения

## 1. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print('Библиотеки успешно загружены')

## 2. Загрузка данных

In [ ]:
# Загружаем данные
df = pd.read_csv('data/eco_tracker_data.csv')
print(f'Форма данных: {df.shape}')
print(f'\nПервые 10 строк:')
print(df.head(10))
print(f'\nИнформация о данных:')
print(df.info())
print(f'\nОписательная статистика:')
print(df.describe())

## 3. Разведочный анализ данных (EDA)

In [ ]:
# Проверка на пропущенные значения
print('Пропущенные значения:')
print(df.isnull().sum())
print(f'\nОбщее количество пропущенных значений: {df.isnull().sum().sum()}')

# Проверка дубликатов
print(f'\nКоличество дубликатов: {df.duplicated().sum()}')

# Распределение целевой переменной
print(f'\nРаспределение целевой переменной (eco_impact_category):')
print(df['eco_impact_category'].value_counts())
print(f'\nПроцентное распределение:')
print(df['eco_impact_category'].value_counts(normalize=True) * 100)

In [ ]:
# Визуализация распределения целевой переменной
fig, ax = plt.subplots(figsize=(10, 6))
df['eco_impact_category'].value_counts().plot(kind='bar', ax=ax, color=['#39FF14', '#00BFFF', '#FFD700', '#FF4500'])
ax.set_title('Распределение экологических категорий влияния', fontsize=14, fontweight='bold')
ax.set_xlabel('Категория экологического влияния', fontsize=12)
ax.set_ylabel('Количество записей', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('График создан')

## 4. Анализ признаков

In [ ]:
# Анализ числовых признаков
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('user_id')  # Убираем ID

print('Числовые признаки для анализа:')
print(numeric_cols)

# Визуализация распределения числовых признаков
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col], bins=30, color='#00BFFF', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Распределение: {col}', fontsize=10, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)

for idx in range(len(numeric_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
print('Графики распределения признаков созданы')

In [ ]:
# Анализ выбросов
print('Анализ выбросов (метод IQR):')
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f'{col}: {len(outliers)} выбросов ({len(outliers)/len(df)*100:.2f}%)')

## 5. Предобработка данных

In [ ]:
# Кодирование категориальных переменных
df_processed = df.copy()

# Кодируем consumption_level
consumption_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
df_processed['consumption_level'] = df_processed['consumption_level'].map(consumption_mapping)

# Кодируем целевую переменную
target_mapping = {'Poor': 0, 'Average': 1, 'Good': 2, 'Excellent': 3}
df_processed['eco_impact_category'] = df_processed['eco_impact_category'].map(target_mapping)

print('Категориальные переменные закодированы:')
print(f'consumption_level: {consumption_mapping}')
print(f'eco_impact_category: {target_mapping}')
print(f'\nПреобразованные данные:')
print(df_processed.head())

In [ ]:
# Удаляем user_id (не используется в моделировании)
X = df_processed.drop(['user_id', 'eco_impact_category'], axis=1)
y = df_processed['eco_impact_category']

print(f'Признаки (X): {X.shape}')
print(f'Целевая переменная (y): {y.shape}')
print(f'\nЛиста признаков:')
print(list(X.columns))

In [ ]:
# Разделение на train, validation и test (60%-20%-20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f'Размеры выборок:')
print(f'Train: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validation: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)')

In [ ]:
# Масштабирование числовых признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Данные масштабированы')
print(f'Среднее значение после масштабирования (train): {X_train_scaled.mean():.4f}')
print(f'Стандартное отклонение после масштабирования (train): {X_train_scaled.std():.4f}')

## 6. Моделирование

In [ ]:
# Модель 1: Random Forest
print('='*60)
print('МОДЕЛЬ 1: Random Forest Classifier')
print('='*60)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

# Предсказания
y_train_pred_rf = rf_model.predict(X_train_scaled)
y_val_pred_rf = rf_model.predict(X_val_scaled)
y_test_pred_rf = rf_model.predict(X_test_scaled)

print('\nРезультаты Random Forest:')
print(f'Train Accuracy: {accuracy_score(y_train, y_train_pred_rf):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, y_val_pred_rf):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_test_pred_rf):.4f}')

print(f'\nTest F1-Score (weighted): {f1_score(y_test, y_test_pred_rf, average="weighted"):.4f}')

In [ ]:
# Модель 2: Gradient Boosting
print('='*60)
print('МОДЕЛЬ 2: Gradient Boosting Classifier')
print('='*60)

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)

# Предсказания
y_train_pred_gb = gb_model.predict(X_train_scaled)
y_val_pred_gb = gb_model.predict(X_val_scaled)
y_test_pred_gb = gb_model.predict(X_test_scaled)

print('\nРезультаты Gradient Boosting:')
print(f'Train Accuracy: {accuracy_score(y_train, y_train_pred_gb):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, y_val_pred_gb):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_test_pred_gb):.4f}')

print(f'\nTest F1-Score (weighted): {f1_score(y_test, y_test_pred_gb, average="weighted"):.4f}')

In [ ]:
# Сравнение моделей
print('='*60)
print('СРАВНЕНИЕ МОДЕЛЕЙ')
print('='*60)

comparison_data = {
    'Метрика': ['Train Accuracy', 'Validation Accuracy', 'Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1-Score'],
    'Random Forest': [
        accuracy_score(y_train, y_train_pred_rf),
        accuracy_score(y_val, y_val_pred_rf),
        accuracy_score(y_test, y_test_pred_rf),
        precision_score(y_test, y_test_pred_rf, average='weighted'),
        recall_score(y_test, y_test_pred_rf, average='weighted'),
        f1_score(y_test, y_test_pred_rf, average='weighted')
    ],
    'Gradient Boosting': [
        accuracy_score(y_train, y_train_pred_gb),
        accuracy_score(y_val, y_val_pred_gb),
        accuracy_score(y_test, y_test_pred_gb),
        precision_score(y_test, y_test_pred_gb, average='weighted'),
        recall_score(y_test, y_test_pred_gb, average='weighted'),
        f1_score(y_test, y_test_pred_gb, average='weighted')
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Выбираем лучшую модель
gb_test_acc = accuracy_score(y_test, y_test_pred_gb)
rf_test_acc = accuracy_score(y_test, y_test_pred_rf)

if gb_test_acc > rf_test_acc:
    print(f'\n✓ ЛУЧШАЯ МОДЕЛЬ: Gradient Boosting (Test Accuracy: {gb_test_acc:.4f})')
    best_model = gb_model
    best_model_name = 'Gradient Boosting'
else:
    print(f'\n✓ ЛУЧШАЯ МОДЕЛЬ: Random Forest (Test Accuracy: {rf_test_acc:.4f})')
    best_model = rf_model
    best_model_name = 'Random Forest'

In [ ]:
# Детальный анализ лучшей модели
if best_model_name == 'Gradient Boosting':
    y_test_pred_best = y_test_pred_gb
else:
    y_test_pred_best = y_test_pred_rf

print(f'\nДЕТАЛЬНЫЙ ОТЧЕТ - {best_model_name}:')
print('='*60)
print(classification_report(y_test, y_test_pred_best, 
                          target_names=['Poor', 'Average', 'Good', 'Excellent']))

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, y_test_pred_best)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Poor', 'Average', 'Good', 'Excellent'],
            yticklabels=['Poor', 'Average', 'Good', 'Excellent'],
            cbar=True, ax=ax)
ax.set_title(f'Матрица ошибок - {best_model_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Предсказанная категория', fontsize=12)
ax.set_ylabel('Истинная категория', fontsize=12)
plt.tight_layout()
plt.show()
print('Матрица ошибок создана')

In [ ]:
# Анализ важности признаков
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\nВажность признаков (Top 10):')
print(feature_importance.head(10).to_string(index=False))

# Визуализация
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(feature_importance)), feature_importance['importance'], color='#00BFFF')
ax.set_yticks(range(len(feature_importance)))
ax.set_yticklabels(feature_importance['feature'])
ax.set_xlabel('Важность признака', fontsize=12)
ax.set_title(f'Важность признаков - {best_model_name}', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
print('\nГрафик важности признаков создан')

## 7. Выводы и рекомендации

In [ ]:
print('ВЫВОДЫ И РЕКОМЕНДАЦИИ')
print('='*60)
print(f'\n1. Лучшая модель: {best_model_name}')
print(f'   - Точность на тестовом наборе: {accuracy_score(y_test, y_test_pred_best):.4f}')
print(f'   - F1-Score: {f1_score(y_test, y_test_pred_best, average="weighted"):.4f}')

print(f'\n2. Ключевые признаки для предсказания:')
for idx, row in feature_importance.head(3).iterrows():
    print(f'   - {row["feature"]}: {row["importance"]:.4f}')

print(f'\n3. Рекомендации:')
print(f'   - Модель готова к развертыванию в production')
print(f'   - Рекомендуется проводить регулярное переобучение на новых данных')
print(f'   - Необходим мониторинг качества модели в production')